# Locale placeholder guard

Check that a translated string catalog kept every `{name}`, every `%s` and every ICU plural
branch the English version started with.

**Not one cell below has been run against the live API.** There was no Sarvam API key on the
machine this recipe was written on, so every code cell ships with an empty output. The two
cells under "The live calls" are the only ones that need a key.

Pipeline:

1. Read `en.json`, the invented demo catalog.
2. Parse each value into text, placeholder and syntax spans.
3. Plan the calls so no value is ever split and no call goes over the character cap.
4. Translate (needs a key).
5. Compare each translation with its source and print a report.

Steps 1 to 3 and step 5 need no key and no network.


In [ ]:
%pip install -r requirements.txt


## Setup

Nothing in this section needs a key.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv

import placeholder_guard as guard

load_dotenv()

CATALOG_PATH = Path("en.json")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))

print(len(catalog), "keys,", sum(len(v) for v in catalog.values()), "characters of text")
for key in list(catalog)[:6]:
    print(f"  {key:<24} {catalog[key]!r}")


## 1. What the grammar sees

`parse` splits a value into spans that tile it: joining the spans back together gives the
original string, character for character. Three kinds of span:

- `text` is what the translator is meant to change.
- `placeholder` must come back byte for byte, though it may move.
- `syntax` is ICU scaffolding: it must survive as a structure, not in place.


In [ ]:
for value in (
    catalog["order.eta_named"],
    catalog["order.battery"],
    catalog["cart.items"],
    catalog["order.number"],
):
    print(repr(value))
    for span in guard.parse(value):
        name = "" if span.name is None else f"  name={span.name}"
        print(f"    {span.kind:<12} {span.text!r:<22} {span.start:>3}..{span.end:<3}{name}")
    print()


Two rules in that output are worth stopping on.

`Delivery van battery at %d%%` holds two placeholders, not one and a stray sign. `%%` prints a
single percent character and consumes no argument, but losing one of its two characters is a
`TypeError` at runtime, so it is checked like any other placeholder.

`Order #{id} is on the way` holds one placeholder. The `#` is ordinary text, because `#` only
means "the number" inside a plural branch. Treating it as a placeholder everywhere would report
a loss for every language that writes the order symbol differently, which is noise.


In [ ]:
print("%d%% ->", guard.placeholder_multiset(catalog["order.battery"]))
print("#{id}  ->", guard.placeholder_multiset(catalog["order.number"]))
print("plural ->", guard.placeholder_multiset(catalog["cart.items"]))


## 2. Why this parses instead of masking a token

The usual way to protect tokens is to replace each one with a numbered sentinel, translate what
is left, and put the tokens back. That is the right shape for a one-off prose message such as a
traceback, where everything inside a protected token is machine text.

It does not work for a catalog. Mask each balanced brace group in the plural below and the
string that reaches the translator is `[[0]] to [[1]]`. The words the user actually reads are
inside the placeholder, and masking makes them unreachable.


In [ ]:
source = "{count, plural, one {# file uploaded} other {# files uploaded}} to {folder}"

spans, out, depth, start = [], [], 0, None
for index, char in enumerate(source):
    if char == "{":
        if depth == 0:
            start = index
        depth += 1
    elif char == "}":
        depth -= 1
        if depth == 0:
            spans.append(source[start:index + 1])
            out.append(f"[[{len(spans) - 1}]]")
    elif depth == 0:
        out.append(char)

print("masked and sent to translate:", repr("".join(out)))
print("words hidden inside span 0  :", spans[0])
print()
print("what this parser hands over :", guard.translatable_text(source))


The parser reaches the branch text and leaves the keywords alone. `count`, `plural`, `one` and
`other` are syntax; `# file uploaded` is text. That distinction is the whole product.


In [ ]:
shapes = guard.icu_shapes(catalog["notify.nested"])
print(shapes[0].name, shapes[0].icu_type, shapes[0].selectors)
for selector, nested in shapes[0].branches:
    print(f"  {selector:<8} -> {[(s.name, s.icu_type, s.selectors) for s in nested]}")
print()
print(guard.translatable_text(catalog["notify.nested"]))


## 3. Plan the calls

`sarvam-translate:v1` accepts 2000 characters of input per call. The planner packs values up to
that cap, and follows four rules in order:

1. A value with no letters in any text span is skipped. `{name}` is not empty and still has
   nothing to translate, so a call for it is money spent to get the same string back.
2. A value longer than the cap on its own is reported, never truncated and never sent.
3. A value that already contains a newline gets its own call, because the newline is the
   separator used to take the reply apart again.
4. Everything else is packed greedily, in catalog order.


In [ ]:
plan = guard.plan_batches(catalog)

print("cap:", guard.TRANSLATE_CHAR_CAP, " batches:", len(plan.batches))
for index, batch in enumerate(plan.batches):
    print(f"  batch {index}: {len(batch.keys):>2} keys, {batch.char_count:>4} chars"
          f"  {batch.keys[0]} .. {batch.keys[-1]}")
for skip in plan.skipped:
    print(f"  skipped: {skip.key} {skip.reason} {skip.char_count}")


At the real cap the whole catalog is three calls, and two of the three boundaries come from
`multiline.address` rather than from length. The interesting behaviour is invisible at 2000
characters, so plan the same catalog again at a much smaller cap.


In [ ]:
small = guard.plan_batches(catalog, cap=200)

print("cap: 200  batches:", len(small.batches))
for index, batch in enumerate(small.batches):
    print(f"  batch {index}: {len(batch.keys):>2} keys, {batch.char_count:>4} chars"
          f"  {batch.keys[0]} .. {batch.keys[-1]}")
for skip in small.skipped:
    print(f"  skipped: {skip.key} {skip.reason} {skip.char_count}")


`notify.nested` is 224 characters, so at a cap of 200 it is reported as `OVER_CAP` and left out
of every batch. A real catalog with a paragraph-length value hits the same rule at 2000. To see
that without shipping two kilobytes of invented prose, repeat a demo value until it is too long.


In [ ]:
oversized = dict(catalog)
oversized["legal.notice"] = (catalog["error.timeout"] + " ") * 60

big = guard.plan_batches(oversized)
for skip in big.skipped:
    print(f"  skipped: {skip.key} {skip.reason} {skip.char_count}")

print()
print("the over-cap value appears in no batch:",
      all("legal.notice" not in batch.keys for batch in big.batches))


## 4. Check a translation, with no key at all

The checker compares the placeholders as a multiset, so word order may change freely -- Hindi
puts the verb last, and asserting the English order would mark every correct translation as
broken. What it will not accept is a placeholder that changed spelling, went missing, appeared
from nowhere, or an ICU argument whose branches are not the ones the source had.

The translations below are hand-written breakages, not API output.


In [ ]:
pairs = [
    ("greeting.named", "hi-IN", "नमस्ते {name}, फिर से स्वागत है"),
    ("greeting.named", "mr-IN", "नमस्कार {नाव}, पुन्हा स्वागत आहे"),
    ("order.driver", "ta-IN", "உங்கள் ஓட்டுநர்"),
    ("order.battery", "hi-IN", "डिलीवरी वैन बैटरी %d"),
    ("cart.items", "hi-IN", "{count, plural, one {# वस्तु} other {# वस्तुएं}} आपकी कार्ट में"),
    ("cart.items", "ta-IN", "{count, plural, one {ஒரு பொருள்} other {# பொருட்கள்}}"),
    ("cart.items", "kn-IN", "{count, plural, other {# ವಸ್ತುಗಳು}}"),
    ("cart.items", "te-IN", "{count, plural, one {# వస్తువు}}"),
    ("notify.gender", "hi-IN",
     "{gender, select, पुरुष {वह} female {वह} other {वे}} ने पार्सल गेट पर छोड़ा"),
]

rows = []
for key, language, translation in pairs:
    check = guard.validate(catalog[key], translation)
    finding = check.findings[0] if check.findings else None
    rows.append(guard.Row(
        key=key,
        language=language,
        verdict=check.verdict,
        placeholders=finding.placeholders if finding else (),
        detail=finding.detail if finding else "",
    ))

print(guard.render_report(rows))


The four `cart.items` rows are the same source string checked against four different
breakages, and they are deliberately four different verdicts:

- `hi-IN` translated the branch words and left the skeleton alone, which is exactly what a
  correct translation of a plural looks like.
- `ta-IN` kept both branches but dropped the `#` out of the `one` branch, so the count would
  not render. That is a lost placeholder: `MISSING`.
- `kn-IN` dropped the `one` branch. What is left is still legal ICU with the wrong shape, so
  it is `SKELETON_CHANGED` rather than a crash.
- `te-IN` dropped the `other` branch. ICU requires `other`, so what is left cannot be
  evaluated at all and does not even parse: `MALFORMED`.

`order.battery` shows the quiet one. The translation kept `%d` and dropped the `%%`, which
looked like a typo, and that alone is a `TypeError` the moment the string is formatted.

Every row also carries a `detail` string that the table leaves out, so a long explanation
cannot wrap the report.


In [ ]:
for row in rows:
    if row.detail:
        print(f"{row.key} / {row.language}: {row.detail}")


## 5. The live calls

**Everything below this line needs a Sarvam API key and has never been run.** The two cells ship
with empty outputs for that reason.

The key is read here, when the cell runs, and passed to the client by hand. The SDK reads the
environment in a default argument, which Python evaluates once at import time, so a client built
without an explicit key fails even when the variable is set afterwards.


In [ ]:
from sarvamai import SarvamAI

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running this cell."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
print("client ready for", guard.TRANSLATE_MODEL, "in", guard.TRANSLATE_MODE, "mode")


`guard_catalog` plans the calls, translates each batch, checks every value that comes back and
returns one row per key per language. If a batch reply does not split into the number of parts
that went out, that batch is retried one value at a time rather than guessed at: a wrong split
would give one key's translation to another key, and nobody would ever notice.

Two languages are used here to keep the call count small. `guard.SCHEDULED_LANGUAGES` holds all
22 that `sarvam-translate:v1` reaches.


In [ ]:
report_rows = guard.guard_catalog(client, catalog, ["hi-IN", "ta-IN"])
report = guard.render_report(report_rows)
print(report)

(OUTPUT_DIR / "placeholder_report.txt").write_text(report, encoding="utf-8")


## What this does not tell you

- It does not say a translation is good. It says the placeholders and the structure survived.
  A string can pass every check here and still be wrong or unidiomatic.
- The demo catalog is invented. It was written to force a decision on every grammar feature,
  not sampled from a real product.
- The grammar reads `%s`, `%d`, `%f` and `%%` only, and ICU `plural` and `select` only. A
  catalog using `%5.2f` or a `date` argument gets a parse error naming the problem, and you
  will have to extend the grammar.
- Nothing here has been run against the live API, so no claim is made about how any of the 22
  languages actually behaves.

Design notes and the full acceptance criteria: `docs/specs/locale-placeholder-guard.md`.
